# Groupby

## Preliminary Note on the Data

Since the data is large (1.5 GB once loaded into a dataframe), we'll define one or more functions per exercise to create our charts.

This way, when each function returns, the temporary variables it used are deleted and the memory can be reused.

## Imports

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import pandas
import seaborn as sns

## Loading the Data

In this lab, we'll use real estate transaction data for all of France, between 2014 and 2022.

This data comes from the [French open data](https://www.data.gouv.fr/fr/datasets/demandes-de-valeurs-foncieres-geolocalisees/) website. We'll use a reworked version that groups transactions, from the [NyxAether/DVF](https://github.com/NyxAether/DVF) repository on GitHub.

In [ ]:
!wget -qO- https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh | bash -s dvf

*You can load the data with the [`pandas.read_parquet`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_parquet.html) function.*

In [ ]:
# Your code here

*Once the data is available in a pandas dataframe, create a `date` column from the `annee_mutation`, `mois_mutation` & `jour_mutation` columns and the [`pandas.to_datetime`](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html) function, then drop these columns to save memory.*

In [ ]:
# Your code here

*Finally, display the memory size and the column types of the dataframe.*

In [ ]:
# Your code here

### Solution

In [ ]:
df = pandas.read_parquet("dvf/dvf-linearized-2014-2022.parquet")

In [ ]:
df["date"] = pandas.to_datetime(
  dict(year=df.annee_mutation, month=df.mois_mutation, day=df.jour_mutation)
)
df.drop(columns=["annee_mutation", "mois_mutation", "jour_mutation"], inplace=True)

For readability later on, we put a ```0``` in front of single-digit departments

In [ ]:
replacedict = {
  "code_departement": {x: "0" + x for x in df.code_departement.unique() if len(x) == 1}
}
df.replace(replacedict, inplace=True)

In [ ]:
df.info()

## Number of Transactions per Year

To start, we'll count the number of transactions (rows of the dataset) per year.

*Use a groupby to count the number of transactions per year*

For nicer results, use a library such as `seaborn` to display bar charts.

In [ ]:
def transactions_by_year() -> None:
  pass  # Your code here


transactions_by_year()

### Solution

In [ ]:
def transactions_by_year() -> None:
  data = df.groupby(df.date.dt.year).valeur_fonciere.count()
  fig, ax = plt.subplots(figsize=(10, 10))
  sns.barplot(x=data.index, y=data, order=data.index, ax=ax)
  sns.despine(ax=ax)
  ax.set_title("Transaction count per year")
  ax.set_xlabel("Year")
  ax.set_ylabel("Number of transactions")
  ax.get_yaxis().set_major_formatter(matplotlib.ticker.EngFormatter(places=1))
  fig.show()


transactions_by_year()

## Average Price per Department

Now, we'll compute the average transaction amount per department.

*Use a groupby to compute the average property value per department*

For nicer results, use a library such as `seaborn` to display bar charts.

In [ ]:
def moyenne_par_departement() -> None:
  pass  # Your code here


moyenne_par_departement()

### Solution

In [ ]:
def moyenne_par_departement() -> None:
  data = (
    df.groupby(df.code_departement).valeur_fonciere.mean().sort_values(ascending=False)
  )

  fig, ax = plt.subplots(figsize=(30, 10))
  sns.barplot(x=data.index, y=data, order=data.index, ax=ax)
  sns.despine(ax=ax)
  ax.set_title("Average transaction amount per department")
  ax.set_xlabel("Department")
  ax.set_ylabel("Average transaction amount")
  ax.get_yaxis().set_major_formatter(matplotlib.ticker.EngFormatter(unit="€", places=1))
  fig.show()


moyenne_par_departement()

In [ ]:
def moyenne_par_departement_std() -> None:
  data = (
    df.groupby(df.code_departement)
    .valeur_fonciere.agg(["min", "mean", "std", "max"])
    .sort_values(by="mean", ascending=False)
  )
  print(data.iloc[:10])
  fig, ax = plt.subplots(figsize=(30, 10))
  sns.barplot(x=data.index, y=data["mean"], order=data.index, ax=ax)
  sns.despine(ax=ax)
  ax.set_title("Average transaction amount per department")
  ax.set_xlabel("Department")
  ax.set_ylabel("Average transaction amount")
  ax.get_yaxis().set_major_formatter(matplotlib.ticker.EngFormatter(unit="€", places=1))
  # for readability, we display the standard deviation divided by 100
  ax.errorbar(
    x=data.index,
    y=data["mean"],
    yerr=data["std"] / 100,
    fmt="none",
    c="black",
    capsize=2,
  )
  fig.show()


moyenne_par_departement_std()

## Average Price by Transaction Type

After looking at the different transaction types (`nature_mutation`), use a groupby to evaluate the average prices for each one. Feel free to display other useful statistical measures.

Then, group these descriptors by department and display them for departments ```75``` and ```23```

In [ ]:
# Your code here

### Solution

In [ ]:
(df.groupby(["nature_mutation"]).valeur_fonciere.describe().style.format(precision=0))

In [ ]:
data = (
  df.groupby(["code_departement", "nature_mutation"])
  .valeur_fonciere.agg(["min", "mean", "std", "max"])
  .sort_values(by="mean", ascending=False)
)
data.loc[["75", "23"]].style.format(precision=0)

## Pivot Tables

Pivot tables and groupby produce (almost) the same result if you don't use pivot_table's `columns` argument

In [ ]:
pt = pandas.pivot_table(
  df,
  index=["code_departement", "nature_mutation"],
  values="valeur_fonciere",
  aggfunc=["min", "mean", "std", "max"],
)
pt.loc[["75", "23"], :].style.format(precision=0)

In [ ]:
pt.loc[["75", "23"]]["mean"]

Notice the difference when using pivot_table's `columns` argument

In [ ]:
pt = pandas.pivot_table(
  df,
  index="code_departement",
  columns="nature_mutation",
  values="valeur_fonciere",
  aggfunc=["min", "mean", "std", "max"],
)
pt.loc[["75", "23"], :].style.format(precision=0)

In [ ]:
pt.loc[["75", "23"]]["mean"]